# Telecom Customer Churn Prediction

This dataset comes from an Iranian telecom company, with each row representing a customer over a year period. Along with a churn label, there is information on the customers' activity, such as call failures and subscription length.

**Business Context:** A competitor has entered the market with an attractive plan for new customers. The telecom company wants to predict which customers are likely to churn, and understand what factors drive that decision.

## Table of Contents
1. [Setup & Data Loading](#1)
2. [Exploratory Data Analysis](#2)
3. [Data Preprocessing](#3)
4. [Feature Engineering](#4)
5. [Model Training & Evaluation](#5)
6. [Feature Importance](#6)
7. [Summary & Business Insights](#7)

## Data Dictionary
| Column                  | Explanation                                             |
|-------------------------|---------------------------------------------------------|
| Call Failure            | number of call failures                                 |
| Complaints              | binary (0: No complaint, 1: complaint)                  |
| Subscription Length     | total months of subscription                            |
| Charge Amount           | ordinal attribute (0: lowest amount, 9: highest amount) |
| Seconds of Use          | total seconds of calls                                  |
| Frequency of use        | total number of calls                                   |
| Frequency of SMS        | total number of text messages                           |
| Distinct Called Numbers | total number of distinct phone calls                    |
| Age Group               | ordinal attribute (1: younger age, 5: older age)        |
| Tariff Plan             | binary (1: Pay as you go, 2: contractual)               |
| Status                  | binary (1: active, 2: non-active)                       |
| Age                     | age of customer                                         |
| Customer Value          | the calculated value of customer                        |
| Churn                   | class label (1: churn, 0: non-churn)                    |

[Source](https://www.kaggle.com/royjafari/customer-churn) of dataset and [source](https://archive.ics.uci.edu/ml/datasets/Iranian+Churn+Dataset) of dataset description.

**Citation**: Jafari-Marandi, R., Denton, J., Idris, A., Smith, B. K., & Keramati, A. (2020). Optimum Profit-Driven Churn Decision Making: Innovative Artificial Neural Networks in Telecom Industry. Neural Computing and Applications.

---
## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix, roc_curve, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid', palette='muted')

churn = pd.read_csv('customer_churn.csv')
print(f'Dataset shape: {churn.shape}')
churn.head()

In [ ]:
print('=== Data Types & Missing Values ===')
info = pd.DataFrame({
    'dtype': churn.dtypes,
    'non_null': churn.notna().sum(),
    'missing': churn.isna().sum(),
    'unique': churn.nunique()
})
print(info)
print(f'\nTotal missing values: {churn.isna().sum().sum()}')

In [ ]:
churn.describe().round(2)

---
## 2. Exploratory Data Analysis <a id='2'></a>

### 2.1 Churn Distribution

In [ ]:
churn_counts = churn['Churn'].value_counts()
churn_pct = churn['Churn'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
bars = axes[0].bar(['No Churn (0)', 'Churn (1)'], churn_counts.values,
                    color=['steelblue', 'coral'], edgecolor='white', linewidth=0.8)
for bar, count, pct in zip(bars, churn_counts.values, churn_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{count:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=11)
axes[0].set_title('Churn Count', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
axes[0].set_ylim(0, churn_counts.max() * 1.15)

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'],
            autopct='%1.1f%%', colors=['steelblue', 'coral'],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Churn Proportion', fontsize=14, fontweight='bold')

plt.suptitle('Customer Churn Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Churn rate: {churn_pct[1]:.1f}% — dataset is moderately imbalanced.')

### 2.2 Numerical Feature Distributions by Churn

In [ ]:
numeric_cols = ['Call Failure', 'Subscription Length', 'Charge Amount',
                'Seconds of Use', 'Frequency of use', 'Frequency of SMS',
                'Distinct Called Numbers', 'Age', 'Customer Value']

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    for label, color in zip([0, 1], ['steelblue', 'coral']):
        data = churn[churn['Churn'] == label][col]
        axes[i].hist(data, bins=25, alpha=0.6, color=color,
                     label='No Churn' if label == 0 else 'Churn', density=True)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].legend(fontsize=9)

plt.suptitle('Numerical Feature Distributions by Churn Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    churn_grouped = [churn[churn['Churn'] == 0][col], churn[churn['Churn'] == 1][col]]
    bp = axes[i].boxplot(churn_grouped, labels=['No Churn', 'Churn'],
                          patch_artist=True, medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][1].set_facecolor('coral')
    for patch in bp['boxes']:
        patch.set_alpha(0.7)
    axes[i].set_title(col, fontsize=11, fontweight='bold')

plt.suptitle('Numerical Feature Boxplots by Churn Status', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.3 Categorical Feature Analysis

In [ ]:
cat_features = {
    'Complaints':    {1: 'Complaint', 0: 'No Complaint'},
    'Age Group':     {1: '1 (Youngest)', 2: '2', 3: '3', 4: '4', 5: '5 (Oldest)'},
    'Tariff Plan':   {1: 'Pay-as-you-go', 2: 'Contractual'},
    'Status':        {1: 'Active', 2: 'Non-active'},
    'Charge Amount': {i: str(i) for i in range(10)}
}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for idx, (col, label_map) in enumerate(cat_features.items()):
    ax = axes[idx]
    ct = churn.groupby(col)['Churn'].agg(['sum', 'count'])
    ct['churn_rate'] = ct['sum'] / ct['count'] * 100
    ct.index = ct.index.map(lambda x: label_map.get(x, str(x)))

    bars = ax.bar(ct.index.astype(str), ct['churn_rate'],
                  color='coral', alpha=0.8, edgecolor='white')
    ax.axhline(churn['Churn'].mean() * 100, color='steelblue',
               linestyle='--', linewidth=1.5, label=f'Overall avg ({churn["Churn"].mean()*100:.1f}%)')
    for bar, rate in zip(bars, ct['churn_rate']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{rate:.1f}%', ha='center', va='bottom', fontsize=9)
    ax.set_title(f'Churn Rate by {col}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, min(105, ct['churn_rate'].max() * 1.25))
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=15)

axes[-1].set_visible(False)
plt.suptitle('Churn Rate by Categorical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.4 Correlation Heatmap

In [ ]:
plt.figure(figsize=(13, 10))
corr = churn.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9})
plt.title('Feature Correlation Matrix', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

churn_corr = corr['Churn'].drop('Churn').sort_values(key=abs, ascending=False)
print('Top correlations with Churn:')
print(churn_corr.to_string())

### 2.5 Challenge Analyses

**Challenge 1:** Which age groups send more SMS messages than make phone calls?

**Challenge 2:** Visualize distinct phone calls by age group, differentiated by call length (short/medium/long).

**Challenge 3:** Are there significant differences in call length between tariff plans?

In [ ]:
# Challenge 1: SMS vs Calls by Age Group
age_group_labels = {1: 'Group 1\n(Youngest)', 2: 'Group 2', 3: 'Group 3',
                    4: 'Group 4', 5: 'Group 5\n(Oldest)'}

agg = churn.groupby('Age Group')[['Frequency of SMS', 'Frequency of use']].mean()
agg.index = agg.index.map(age_group_labels)

fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(agg))
w = 0.35
bars1 = ax.bar(x - w/2, agg['Frequency of SMS'], w, label='Avg SMS', color='mediumorchid', alpha=0.85)
bars2 = ax.bar(x + w/2, agg['Frequency of use'], w, label='Avg Calls', color='steelblue', alpha=0.85)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(agg.index)
ax.set_title('Average SMS vs Phone Calls by Age Group', fontsize=14, fontweight='bold')
ax.set_ylabel('Average Count per Customer')
ax.legend()
plt.tight_layout()
plt.show()

sms_dominant = agg[agg['Frequency of SMS'] > agg['Frequency of use']].index.tolist()
print(f'Age groups where SMS > Calls: {sms_dominant if sms_dominant else "None"}')
print('\nSMS vs Calls breakdown:')
print(agg.round(2))

In [ ]:
# Challenge 2: Distinct called numbers by age group, split by call duration bucket
# Bucket by avg call duration (Seconds of Use / Frequency of use)
df_c2 = churn.copy()
df_c2['Avg Call Secs'] = df_c2['Seconds of Use'] / df_c2['Frequency of use'].replace(0, np.nan)

# Define short/medium/long using tertiles
short_thresh = df_c2['Avg Call Secs'].quantile(0.33)
long_thresh  = df_c2['Avg Call Secs'].quantile(0.67)

def call_bucket(s):
    if pd.isna(s):  return 'No Calls'
    if s <= short_thresh: return 'Short'
    if s <= long_thresh:  return 'Medium'
    return 'Long'

df_c2['Call Length'] = df_c2['Avg Call Secs'].apply(call_bucket)
df_c2['Age Group Label'] = df_c2['Age Group'].map({1:'G1 (Youngest)',2:'G2',3:'G3',4:'G4',5:'G5 (Oldest)'})

pivot = df_c2.groupby(['Age Group Label', 'Call Length'])['Distinct Called Numbers'].mean().unstack(fill_value=0)
# Reorder columns
col_order = [c for c in ['Short', 'Medium', 'Long', 'No Calls'] if c in pivot.columns]
pivot = pivot[col_order]

ax = pivot.plot(kind='bar', stacked=True, figsize=(12, 6),
                color=['#66c2a5', '#fc8d62', '#8da0cb', '#e78ac3'],
                edgecolor='white', linewidth=0.5)
ax.set_title('Avg Distinct Called Numbers by Age Group & Call Length', fontsize=14, fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Avg Distinct Called Numbers')
ax.legend(title='Call Length', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(f'Short call threshold:  <= {short_thresh:.0f} seconds')
print(f'Medium call threshold: <= {long_thresh:.0f} seconds')
print(f'Long calls:            >  {long_thresh:.0f} seconds')

In [ ]:
# Challenge 3: Call duration differences between tariff plans
df_c3 = churn.copy()
df_c3['Avg Call Secs'] = df_c3['Seconds of Use'] / df_c3['Frequency of use'].replace(0, np.nan)
df_c3['Tariff Label'] = df_c3['Tariff Plan'].map({1: 'Pay-as-you-go', 2: 'Contractual'})

payg = df_c3[df_c3['Tariff Plan'] == 1]['Avg Call Secs'].dropna()
contract = df_c3[df_c3['Tariff Plan'] == 2]['Avg Call Secs'].dropna()

from scipy import stats
t_stat, p_value = stats.mannwhitneyu(payg, contract, alternative='two-sided')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Violin plot
parts = axes[0].violinplot([payg, contract], positions=[1, 2],
                            showmedians=True, showmeans=False)
for i, (pc, color) in enumerate(zip(parts['bodies'], ['steelblue', 'coral'])):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(['Pay-as-you-go', 'Contractual'])
axes[0].set_title('Call Duration Distribution by Tariff Plan', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Avg Call Duration (seconds)')

# Box plot with jitter sample
sample_payg = payg.sample(min(300, len(payg)), random_state=42)
sample_contract = contract.sample(min(300, len(contract)), random_state=42)
axes[1].boxplot([payg, contract], labels=['Pay-as-you-go', 'Contractual'],
                patch_artist=True,
                boxprops=dict(facecolor='lightgray'),
                medianprops=dict(color='black', linewidth=2))
axes[1].scatter(np.random.normal(1, 0.05, len(sample_payg)), sample_payg,
                alpha=0.3, s=8, color='steelblue')
axes[1].scatter(np.random.normal(2, 0.05, len(sample_contract)), sample_contract,
                alpha=0.3, s=8, color='coral')
axes[1].set_title('Call Duration Boxplot by Tariff Plan', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Avg Call Duration (seconds)')

plt.suptitle(f'Mann-Whitney U test: p = {p_value:.4f} — {"Significant" if p_value < 0.05 else "Not significant"} difference',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Pay-as-you-go  — median: {payg.median():.0f}s, mean: {payg.mean():.0f}s')
print(f'Contractual    — median: {contract.median():.0f}s, mean: {contract.mean():.0f}s')
print(f'Mann-Whitney U statistic: {t_stat:.0f}, p-value: {p_value:.4f}')
sig = 'IS' if p_value < 0.05 else 'is NOT'
print(f'\nConclusion: There {sig} a statistically significant difference in call duration between tariff plans (alpha=0.05).')

---
## 3. Data Preprocessing <a id='3'></a>

In [ ]:
df = churn.copy()

# Tariff Plan: 1=pay-as-you-go, 2=contractual → encode as 0/1 (1=contractual)
df['Tariff Plan'] = (df['Tariff Plan'] == 2).astype(int)

# Status: 1=active, 2=non-active → encode as 0/1 (1=non-active)
df['Status'] = (df['Status'] == 2).astype(int)

# All other features are already numeric — no missing values, no encoding needed
print('Preprocessed data types:')
print(df.dtypes)
print(f'\nMissing values: {df.isna().sum().sum()}')
print(f'Shape: {df.shape}')
df.head()

---
## 4. Feature Engineering <a id='4'></a>

In [ ]:
# Avg call duration (seconds per call)
df['Avg Call Duration'] = df['Seconds of Use'] / df['Frequency of use'].replace(0, np.nan)
df['Avg Call Duration'] = df['Avg Call Duration'].fillna(0)

# SMS-to-calls ratio: preference for text vs voice
total_comm = df['Frequency of SMS'] + df['Frequency of use']
df['SMS Ratio'] = df['Frequency of SMS'] / total_comm.replace(0, np.nan)
df['SMS Ratio'] = df['SMS Ratio'].fillna(0)

# Call failure rate (failures per call made)
df['Failure Rate'] = df['Call Failure'] / df['Frequency of use'].replace(0, np.nan)
df['Failure Rate'] = df['Failure Rate'].fillna(0)

# Subscription tenure group: new (<=12m), mid (12-36m), loyal (>36m)
df['Tenure Group'] = pd.cut(df['Subscription Length'], bins=[0, 12, 36, 72],
                             labels=[0, 1, 2], include_lowest=True).astype(int)

print(f'New features added. Total features: {df.shape[1]}')
print('New columns:', ['Avg Call Duration', 'SMS Ratio', 'Failure Rate', 'Tenure Group'])
df[['Avg Call Duration', 'SMS Ratio', 'Failure Rate', 'Tenure Group']].describe().round(3)

---
## 5. Model Training & Evaluation <a id='5'></a>

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Training set: {X_train.shape[0]} rows | Test set: {X_test.shape[0]} rows')
print(f'Training churn rate: {y_train.mean()*100:.1f}%')
print(f'Test churn rate:     {y_test.mean()*100:.1f}%')

# Apply SMOTE to training set to address class imbalance
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'\nAfter SMOTE — training set: {X_train_res.shape[0]} rows')
print(f'Resampled churn rate: {y_train_res.mean()*100:.1f}%')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
models = {
    'Logistic Regression (L1)': LogisticRegression(penalty='l1', solver='liblinear',
                                                     C=0.5, max_iter=500, random_state=42),
    'Logistic Regression (L2)': LogisticRegression(penalty='l2', solver='lbfgs',
                                                     max_iter=500, random_state=42),
    'Decision Tree':            DecisionTreeClassifier(max_depth=6, random_state=42),
    'KNN':                      KNeighborsClassifier(n_neighbors=7),
    'Random Forest':            RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting':        GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, random_state=42),
    'XGBoost':                  xgb.XGBClassifier(n_estimators=200, learning_rate=0.05,
                                                   eval_metric='logloss', random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train_res)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = {
        'model':     model,
        'y_pred':    y_pred,
        'y_prob':    y_prob,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall':    recall_score(y_test, y_pred),
        'F1':        f1_score(y_test, y_pred),
        'ROC-AUC':   roc_auc_score(y_test, y_prob)
    }
    print(f'{name:<28} | Acc: {results[name]["Accuracy"]:.3f} | F1: {results[name]["F1"]:.3f} | AUC: {results[name]["ROC-AUC"]:.3f}')

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
results_df = pd.DataFrame(
    {name: {m: results[name][m] for m in metrics} for name in results}
).T.sort_values('ROC-AUC', ascending=False)

print('Model Comparison (sorted by ROC-AUC):')
print(results_df.round(4).to_string())

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(results_df))
width = 0.15
colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

for i, (metric, color) in enumerate(zip(metrics, colors)):
    bars = ax.bar(x + i * width, results_df[metric], width,
                  label=metric, color=color, alpha=0.85)

ax.set_xticks(x + width * 2)
ax.set_xticklabels(results_df.index, rotation=15, ha='right')
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', ncol=5)
ax.axhline(0.8, color='gray', linestyle=':', linewidth=1, alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
colors_roc = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0', '#00BCD4']

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f"{name} (AUC={res['ROC-AUC']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices for the top 2 models by ROC-AUC
top2 = results_df.head(2).index.tolist()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name in zip(axes, top2):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAUC={results[name]["ROC-AUC"]:.3f}  F1={results[name]["F1"]:.3f}',
                 fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices — Top 2 Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
best_name = results_df.index[0]
best = results[best_name]
print(f'Best model: {best_name}')
print('=' * 60)
print(classification_report(y_test, best['y_pred'], target_names=['No Churn', 'Churn']))

### 5.2 Hyperparameter Tuning — Decision Tree\n\nFollowing the DataCamp tutorial approach, we tune the Decision Tree's `max_depth` and `min_samples_leaf` to reduce overfitting and find the optimal tree complexity.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth':        [3, 4, 5, 6, 8, 10, None],
    'min_samples_leaf': [1, 5, 10, 20],
    'criterion':        ['gini', 'entropy']
}

dt_cv = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1
)
dt_cv.fit(X_train_scaled, y_train_res)

print('Best Decision Tree parameters:', dt_cv.best_params_)
print(f'Best CV ROC-AUC: {dt_cv.best_score_:.3f}')

# Evaluate tuned DT on test set
y_pred_dt = dt_cv.predict(X_test_scaled)
y_prob_dt = dt_cv.predict_proba(X_test_scaled)[:, 1]
print(f'\nTuned Decision Tree — Test Set:')
print(f'  ROC-AUC:  {roc_auc_score(y_test, y_prob_dt):.3f}')
print(f'  F1 Score: {f1_score(y_test, y_pred_dt):.3f}')
print(f'  Recall:   {recall_score(y_test, y_pred_dt):.3f}')

# Compare max_depth vs AUC
depth_scores = {}
for depth in [2, 3, 4, 5, 6, 8, 10, 15]:
    dt_tmp = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_tmp.fit(X_train_scaled, y_train_res)
    prob = dt_tmp.predict_proba(X_test_scaled)[:, 1]
    depth_scores[depth] = roc_auc_score(y_test, prob)

plt.figure(figsize=(9, 5))
plt.plot(list(depth_scores.keys()), list(depth_scores.values()),
         marker='o', color='steelblue', linewidth=2, markersize=7)
plt.axvline(dt_cv.best_params_['max_depth'], color='coral', linestyle='--',
            linewidth=1.5, label=f'Best depth = {dt_cv.best_params_[\"max_depth\"]}')
plt.xlabel('max_depth', fontsize=12)
plt.ylabel('Test ROC-AUC', fontsize=12)
plt.title('Decision Tree: max_depth vs ROC-AUC', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## 6. Feature Importance <a id='6'></a>

In [ ]:
# Tree-based feature importance (Random Forest or Gradient Boosting)
tree_models = ['Random Forest', 'Gradient Boosting', 'XGBoost', 'Decision Tree']
best_tree_name = next((m for m in results_df.index if m in tree_models), None)

if best_tree_name:
    best_tree = results[best_tree_name]['model']
    importances = best_tree.feature_importances_
    feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

    plt.figure(figsize=(11, 7))
    colors_fi = ['coral' if i < 5 else 'steelblue' for i in range(len(feat_imp[:15]))]
    bars = plt.barh(feat_imp[:15].index[::-1], feat_imp[:15].values[::-1],
                    color=colors_fi[::-1], alpha=0.85, edgecolor='white')
    plt.xlabel('Feature Importance Score', fontsize=12)
    plt.title(f'Top 15 Feature Importances — {best_tree_name}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f'Top 10 features ({best_tree_name}):')
    print(feat_imp.head(10).round(4).to_string())

In [ ]:
# Logistic Regression (L1) — churn driver analysis
# L1 regularization zeros out weak features, so non-zero coefficients are the true drivers
lr_l1 = results['Logistic Regression (L1)']['model']
coef_series = pd.Series(lr_l1.coef_[0], index=X.columns)

# Separate features zeroed out by L1
nonzero = coef_series[coef_series != 0].sort_values(key=abs, ascending=False)
zeroed  = coef_series[coef_series == 0]

print(f'L1 zeroed out {len(zeroed)} features: {zeroed.index.tolist()}')
print(f'Active churn drivers ({len(nonzero)} features):')
print(nonzero.round(4).to_string())

fig, ax = plt.subplots(figsize=(11, 7))
colors_coef = ['coral' if v > 0 else 'steelblue' for v in nonzero.values[::-1]]
ax.barh(nonzero.index[::-1], nonzero.values[::-1], color=colors_coef, alpha=0.85, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (positive = increases churn risk)', fontsize=12)
ax.set_title('Churn Drivers — Logistic Regression with L1 Regularization\n(Coral = increases churn risk, Blue = decreases churn risk)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Summary & Business Insights <a id='7'></a>

### Key Findings

| Driver | Insight |
|---|---|
| **Complaints** | Customers who filed a complaint churn at a far higher rate — the strongest single indicator |
| **Status** | Non-active customers almost always churn — status is a near-perfect proxy |
| **Subscription Length** | Shorter subscriptions correlate with higher churn; loyal customers rarely leave |
| **Call Failure Rate** | High call failure rates frustrate customers and drive churn |
| **Age Group** | Younger customers (Group 1) churn more; older groups are more loyal |
| **Tariff Plan** | Pay-as-you-go customers churn more than contractual customers |
| **Customer Value** | Lower-value customers are more likely to churn |

### Actionable Recommendations

1. **Complaint Resolution Program** — Proactively reach out to customers who file complaints with immediate resolution offers or retention discounts. Complaint history is the strongest churn signal.

2. **Early Engagement for New Customers** — Customers in the first 12 months of subscription churn most. Implement onboarding check-ins and loyalty incentives for the first year.

3. **Network Quality Investment** — High call failure rates directly drive dissatisfaction. Prioritize infrastructure improvements in regions with elevated failure rates.

4. **Contract Upgrade Offers** — Pay-as-you-go customers churn more than contractual ones. Offer attractive conversion packages to move them onto annual contracts.

5. **Targeted Retention for Young Customers** — Age Group 1 customers are most at risk. Tailor retention campaigns (e.g., SMS-heavy plans, competitive pricing) to younger demographics.

6. **Proactive Monitoring of Non-Active Status** — Customers moving to non-active status should trigger immediate outreach before they formally churn.

In [ ]:
print('=' * 65)
print('CUSTOMER CHURN PREDICTION — FINAL MODEL SUMMARY')
print('=' * 65)
print(f'Dataset:    {churn.shape[0]:,} customers, {churn.shape[1]} features')
print(f'Churn rate: {churn["Churn"].mean()*100:.1f}% (class imbalance handled with SMOTE)')
print()
print('Model Performance (Test Set):')
print('-' * 65)
for name in results_df.index:
    r = results[name]
    marker = ' <-- BEST' if name == best_name else ''
    print(f"  {name:<22} | AUC={r['ROC-AUC']:.3f} | F1={r['F1']:.3f} | Recall={r['Recall']:.3f}{marker}")
print('-' * 65)
print(f'\nBest model: {best_name}')
print(f'  ROC-AUC:  {results[best_name]["ROC-AUC"]:.3f}')
print(f'  F1 Score: {results[best_name]["F1"]:.3f}')
print(f'  Recall:   {results[best_name]["Recall"]:.3f}  (correctly identifies {results[best_name]["Recall"]*100:.1f}% of churners)')
print(f'  Accuracy: {results[best_name]["Accuracy"]:.3f}')